# Membrane–actin Brownian ratchet — from boundary baselines to a coupled flexible membrane

_Investigation `membrane-actin-ratchet` — coder reproduction notebook._

**Question.** Can a particle-based actin Brownian ratchet (ReaDDy) push against a deformable membrane
(Mem3DG) through a physically faithful bidirectional coupler, and does increasing boundary
realism — fixed wall → rigid movable object → flexible membrane — reproduce the
force/velocity signatures that distinguish a flexible-membrane ratchet (Inoue 2015) from
the classical rigid-wall regime (Peskin 1993)?

An investigation is a sequence of studies that together answer one research question. This
one asks whether a coupled actin (ReaDDy) + membrane (Mem3DG) composite behaves like a real
flexible-membrane Brownian ratchet, by comparing three boundary models and stress-testing the
coupler that powers the flexible one. It merges two earlier framings — a boundary-condition
staircase and a coupling-mechanism deep-dive — into one arc, because the staircase's top rung
*is* the coupler the deep-dive validates.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-membrane-actin-composite/viva-membrane-actin-composite').is_dir():
    REPO = Path('/home/runner/work/viva-membrane-actin-composite/viva-membrane-actin-composite')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_membrane_actin_composite.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `fixed-boundary`

**Question.** Does the composite reproduce the rigid-wall Peskin-1993 regime — Brownian-ratchet kinetics against an immovable boundary — to within numerical tolerance, when barrier_kind is "fixed"?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fixed-boundary` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 5 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 5 | growth_rate=8.0 |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 5 | growth_rate=1.0 |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_membrane_actin_composite.composites.rung1_fixed_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/pbg_membrane_actin_composite.composites.rung1_fixed_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fixed-boundary ===
STUDY = 'fixed-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_fixed_boundary = 5
INTERVAL_fixed_boundary = 0.1
STEPS_high_growth = 5
INTERVAL_high_growth = 0.1
STEPS_low_growth = 5
INTERVAL_low_growth = 0.1
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 0.1
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 0.1
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 0.1
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 0.1
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_fixed_boundary}, core=core)
        comp.run(STEPS_fixed_boundary)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_high_growth}, core=core)
        comp.run(STEPS_high_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_low_growth}, core=core)
        comp.run(STEPS_low_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
        comp.run(STEPS_rung1_fixed)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
        comp.run(STEPS_rung1_fixed)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
        comp.run(STEPS_rung1_fixed)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
        comp.run(STEPS_rung1_fixed)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
        comp.run(STEPS_rung1_fixed)  # writes the composite's declared emitter
    print(f'ran 8 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**gap-contact-decision**


In [ ]:
# gap-contact-decision
show_viz(_render_one('local:GapContactDecision', {'study_slug': 'fixed-boundary'}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'fixed-boundary'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fv-matches-mogilner-baseline | kind=listener_path observable=barrier_velocity reduce=mean | op in_range lo -0.001 hi 0.001 |


## Study: `rigid-movable-boundary`

**Question.** With barrier_kind="rigid_movable", does the wall translate at a velocity proportional to the mean contact force (drag-balance Peskin 1993 setup), and does the stall force remain bounded?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `high-drag` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 5 | barrier_drag=16.0 |
| `low-drag` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 5 | barrier_drag=2.0 |
| `rigid-movable-boundary` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 5 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: rigid-movable-boundary ===
STUDY = 'rigid-movable-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_high_drag = 5
INTERVAL_high_drag = 0.1
STEPS_low_drag = 5
INTERVAL_low_drag = 0.1
STEPS_rigid_movable_boundary = 5
INTERVAL_rigid_movable_boundary = 0.1
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 0.1
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 0.1
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 0.1
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 0.1
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_high_drag}, core=core)
        comp.run(STEPS_high_drag)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_low_drag}, core=core)
        comp.run(STEPS_low_drag)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rigid_movable_boundary}, core=core)
        comp.run(STEPS_rigid_movable_boundary)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
        comp.run(STEPS_rung2_rigid_movable)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
        comp.run(STEPS_rung2_rigid_movable)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
        comp.run(STEPS_rung2_rigid_movable)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
        comp.run(STEPS_rung2_rigid_movable)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
        comp.run(STEPS_rung2_rigid_movable)  # writes the composite's declared emitter
    print(f'ran 8 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fv-phase-portrait**


In [ ]:
# fv-phase-portrait
show_viz(_render_one('local:FVPhasePortrait', {'study_slug': 'rigid-movable-boundary'}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**force-velocity-scatter**


In [ ]:
# force-velocity-scatter
show_viz(_render_one('local:ForceVelocityScatter', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'rigid-movable-boundary'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| stall-force-within-peskin-bound | kind=listener_path observable=barrier_velocity reduce=mean | op in_range lo 0.001 hi 5.0 |


## Study: `actin-to-membrane-force-handshake`

**Question.** In the closed-loop flexible composite, is the force the actin side reports at the interface equal-and-opposite to the force the membrane integrates on its receiving vertices (Newton's 3rd law on the coupling interface)?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: actin-to-membrane-force-handshake ===
STUDY = 'actin-to-membrane-force-handshake'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**newton-residual**


In [ ]:
# newton-residual
show_viz(_render_one('local:NewtonResidual', {'study_slug': 'actin-to-membrane-force-handshake'}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**backpressure-trace**


In [ ]:
# backpressure-trace
show_viz(_render_one('local:BackpressureTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'actin-to-membrane-force-handshake'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| newton-third-law-holds-on-interface | kind=listener_path observable=contact_force reduce=series | op rolling_cv_below threshold 0.05 |


## Study: `membrane-to-actin-displacement-feedback`

**Question.** Does per-step mesh-position feedback from Mem3DG to ReaDDy preserve mesh quality (no vertex inversion or degenerate triangles) and keep tip particles within their box-potential bounds?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `aggressive-feedback` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.12 |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `gentle-feedback` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.02 |
| `membrane-to-actin-displacement-feedback` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: membrane-to-actin-displacement-feedback ===
STUDY = 'membrane-to-actin-displacement-feedback'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_aggressive_feedback = 5
INTERVAL_aggressive_feedback = 0.1
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 0.1
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 0.1
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 0.1
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 0.1
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 0.1
STEPS_gentle_feedback = 5
INTERVAL_gentle_feedback = 0.1
STEPS_membrane_to_actin_displacement_feedback = 5
INTERVAL_membrane_to_actin_displacement_feedback = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_aggressive_feedback}, core=core)
        comp.run(STEPS_aggressive_feedback)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
        comp.run(STEPS_feedback_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
        comp.run(STEPS_feedback_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
        comp.run(STEPS_feedback_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
        comp.run(STEPS_feedback_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
        comp.run(STEPS_feedback_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_gentle_feedback}, core=core)
        comp.run(STEPS_gentle_feedback)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_membrane_to_actin_displacement_feedback}, core=core)
        comp.run(STEPS_membrane_to_actin_displacement_feedback)  # writes the composite's declared emitter
    print(f'ran 8 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**backpressure-trace**


In [ ]:
# backpressure-trace
show_viz(_render_one('local:BackpressureTrace', {}, RUNS_DB, STUDY_YAML))

**membrane-volume-strain**


In [ ]:
# membrane-volume-strain
show_viz(_render_one('local:MembraneVolumeStrain', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'membrane-to-actin-displacement-feedback'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mesh-quality-preserved-over-coupling-steps | kind=listener_path observable=membrane_min_z reduce=series | op monotonic_decreasing |


## Study: `flexible-mem3dg-boundary`

**Question.** With barrier_kind="flexible" (closed-icosphere Mem3DG mesh inside-pushed by bonded actin filaments), does the membrane deform under actin pressure and relax the contact-force burden compared to the rigid_movable rung?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `flexible-mem3dg-boundary` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `soft-coupling` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.02 |
| `stiff-coupling` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.1 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: flexible-mem3dg-boundary ===
STUDY = 'flexible-mem3dg-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_flexible_mem3dg_boundary = 5
INTERVAL_flexible_mem3dg_boundary = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 0.1
STEPS_soft_coupling = 5
INTERVAL_soft_coupling = 0.1
STEPS_stiff_coupling = 5
INTERVAL_stiff_coupling = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_flexible_mem3dg_boundary}, core=core)
        comp.run(STEPS_flexible_mem3dg_boundary)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
        comp.run(STEPS_rung3_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_soft_coupling}, core=core)
        comp.run(STEPS_soft_coupling)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_stiff_coupling}, core=core)
        comp.run(STEPS_stiff_coupling)  # writes the composite's declared emitter
    print(f'ran 11 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**backpressure-trace**


In [ ]:
# backpressure-trace
show_viz(_render_one('local:BackpressureTrace', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**membrane-volume-strain**


In [ ]:
# membrane-volume-strain
show_viz(_render_one('local:MembraneVolumeStrain', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**force-velocity-scatter**


In [ ]:
# force-velocity-scatter
show_viz(_render_one('local:ForceVelocityScatter', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'flexible-mem3dg-boundary'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| membrane-curvature-relaxes-load | kind=listener_path observable=membrane_volume reduce=first_and_last | op ratio_at_least ratio 1.1 |


## Study: `coupled-ratchet-cycle-fv-reproduction`

**Question.** Does the closed-loop coupled cycle reproduce the Inoue-2015 concave→convex F-V transition as polymerization rate varies — the headline numerical benchmark for a flexible-membrane ratchet (spec §1.2)?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-ratchet-cycle-fv-reproduction` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | growth_rate=8.0 |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | growth_rate=1.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupled-ratchet-cycle-fv-reproduction ===
STUDY = 'coupled-ratchet-cycle-fv-reproduction'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 0.1
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 0.1
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 0.1
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 0.1
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 0.1
STEPS_coupled_ratchet_cycle_fv_reproduction = 5
INTERVAL_coupled_ratchet_cycle_fv_reproduction = 0.1
STEPS_high_growth = 16
INTERVAL_high_growth = 0.1
STEPS_high_growth = 16
INTERVAL_high_growth = 0.1
STEPS_high_growth = 16
INTERVAL_high_growth = 0.1
STEPS_high_growth = 16
INTERVAL_high_growth = 0.1
STEPS_high_growth = 5
INTERVAL_high_growth = 0.1
STEPS_low_growth = 16
INTERVAL_low_growth = 0.1
STEPS_low_growth = 16
INTERVAL_low_growth = 0.1
STEPS_low_growth = 16
INTERVAL_low_growth = 0.1
STEPS_low_growth = 16
INTERVAL_low_growth = 0.1
STEPS_low_growth = 5
INTERVAL_low_growth = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
        comp.run(STEPS_coupled_cycle_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
        comp.run(STEPS_coupled_cycle_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
        comp.run(STEPS_coupled_cycle_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
        comp.run(STEPS_coupled_cycle_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
        comp.run(STEPS_coupled_cycle_flexible)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_ratchet_cycle_fv_reproduction}, core=core)
        comp.run(STEPS_coupled_ratchet_cycle_fv_reproduction)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
        comp.run(STEPS_high_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
        comp.run(STEPS_high_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
        comp.run(STEPS_high_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
        comp.run(STEPS_high_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
        comp.run(STEPS_high_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
        comp.run(STEPS_low_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
        comp.run(STEPS_low_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
        comp.run(STEPS_low_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
        comp.run(STEPS_low_growth)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
        comp.run(STEPS_low_growth)  # writes the composite's declared emitter
    print(f'ran 16 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**force-velocity-benchmark**


In [ ]:
# force-velocity-benchmark
show_viz(_render_one('local:ForceVelocityBenchmark', {'study_slug': 'coupled-ratchet-cycle-fv-reproduction'}, RUNS_DB, STUDY_YAML))

**force-velocity-scatter**


In [ ]:
# force-velocity-scatter
show_viz(_render_one('local:ForceVelocityScatter', {}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**membrane-volume-strain**


In [ ]:
# membrane-volume-strain
show_viz(_render_one('local:MembraneVolumeStrain', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'coupled-ratchet-cycle-fv-reproduction'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| concave-to-convex-fv-transition-emerges | kind=listener_path observable=barrier_velocity reduce=mean | op ratio_at_least ratio 2.0 compared_to {'run': 'variant', 'variant': 'low-growth'} |


## Open decisions
- Which numerical tolerances define "matches Peskin / Inoue"?
- Spherical mesh-vertex feedback or planar wall_radius for the canonical F-V run?
- Where do the Peskin (1993) and Inoue (2015) reference curves come from for the overlays?
